In [ ]:
# Cell 1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
print("Drive mounted")
print("Contents:", os.listdir('/content/drive/MyDrive'))

Mounted at /content/drive
Drive mounted
Contents: ['VIGHNESH DOC', 'mahadbt2025-26.pdf', 'Colab Notebooks', 'deepfake', 'deefake_detection_system', 'deep']


In [ ]:
# Cell 2 — Paths
# ── Update SOURCE_DIR to match your exact Drive path ──────────────────────
SOURCE_DIR = "/content/drive/MyDrive/deepfake/dataset/data"
OUTPUT_DIR = "/content/drive/MyDrive/deepfake/dataset/split"

# Verify source exists
import os
print("Source exists :", os.path.exists(SOURCE_DIR))
print("Source folders:", os.listdir(SOURCE_DIR))
# Expected output: ['fake', 'real']

Source exists : True
Source folders: ['fake', 'real']


In [ ]:
# Cell 3 — Imports
import os, shutil, random
from pathlib import Path
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

PyTorch  : 2.10.0+cu128
CUDA     : True
GPU      : Tesla T4


In [ ]:
# Cell 4 — Split function
def split_dataset(source_dir, output_dir, split_ratio=0.8, seed=42):
    random.seed(seed)
    classes = ["fake", "real"]
    stats   = {}

    for cls in classes:
        src = Path(source_dir) / cls
        if not src.exists():
            raise FileNotFoundError(f"Not found: {src}")

        images = sorted([
            f for f in src.iterdir()
            if f.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
        ])
        if not images:
            raise ValueError(f"No images in {src}")

        random.shuffle(images)
        n_train     = int(len(images) * split_ratio)
        train_files = images[:n_train]
        val_files   = images[n_train:]

        train_out = Path(output_dir) / "train" / cls
        val_out   = Path(output_dir) / "val"   / cls
        train_out.mkdir(parents=True, exist_ok=True)
        val_out.mkdir(parents=True,   exist_ok=True)

        print(f"Copying {cls} train ({len(train_files)})...", end=" ")
        for f in train_files:
            shutil.copy2(f, train_out / f.name)
        print("done")

        print(f"Copying {cls} val   ({len(val_files)})...", end=" ")
        for f in val_files:
            shutil.copy2(f, val_out / f.name)
        print("done")

        stats[cls] = {
            "total": len(images),
            "train": len(train_files),
            "val":   len(val_files),
        }

    print("\n" + "=" * 44)
    print(f"  {'CLASS':<6} | {'TOTAL':>6} | {'TRAIN':>6} | {'VAL':>6}")
    print("=" * 44)
    for cls, s in stats.items():
        print(f"  {cls:<6} | {s['total']:>6} | {s['train']:>6} | {s['val']:>6}")
    total_tr = sum(s["train"] for s in stats.values())
    total_v  = sum(s["val"]   for s in stats.values())
    print("=" * 44)
    print(f"  {'TOTAL':<6} | {total_tr+total_v:>6} | {total_tr:>6} | {total_v:>6}")
    print(f"\nSplit saved to: {output_dir}")
    return stats

print("Split function ready")

Split function ready


In [ ]:
# Cell 5 — Run split
# NOTE: This copies files — takes 3-8 mins depending on Drive speed
# Do NOT interrupt once started

stats = split_dataset(
    source_dir = SOURCE_DIR,
    output_dir = OUTPUT_DIR,
    split_ratio = 0.8,
    seed = 42,
)


Copying fake train (4400)... done
Copying fake val   (1100)... done
Copying real train (4400)... done
Copying real val   (1100)... done

  CLASS  |  TOTAL |  TRAIN |    VAL
  fake   |   5500 |   4400 |   1100
  real   |   5500 |   4400 |   1100
  TOTAL  |  11000 |   8800 |   2200

Split saved to: /content/drive/MyDrive/deepfake/dataset/split


In [ ]:
# Cell 6 — Verify
splits  = ["train", "val"]
classes = ["fake", "real"]
all_ok  = True

print("Verifying output structure...\n")
for split in splits:
    for cls in classes:
        path  = f"{OUTPUT_DIR}/{split}/{cls}"
        files = [f for f in os.listdir(path)
                 if f.lower().endswith((".jpg",".jpeg",".png",".webp"))]
        status = "OK" if files else "EMPTY"
        print(f"  [{status}] {split}/{cls}: {len(files)} images")
        if not files:
            all_ok = False

print()
print("Verification:", "PASSED" if all_ok else "FAILED")

Verifying output structure...

  [OK] train/fake: 4400 images
  [OK] train/real: 4400 images
  [OK] val/fake: 1100 images
  [OK] val/real: 1100 images

Verification: PASSED
